In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
KEY_PATH = "Key_Path"
SOURCE_DIR = "/content/drive/MyDrive/Portfolio Attachments/AICallsSystemData_copy"
DATASET_NAME = "callva_data"
TABLE_NAME = "master_campaign_analytics_copy"
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
table_id = f"{credentials.project_id}.{DATASET_NAME}.{TABLE_NAME}"

def get_company_name(campaign_name):
    """
    Extracts the company name from the filename.
    Pattern: Leads_CompanyName_SomethingElse -> returns 'CompanyName'
    """
    parts = campaign_name.split('_')
    if len(parts) >= 2:
        return parts[1]
    return "General"

print(f"Scanning for updates in: {SOURCE_DIR}")
print(f"Target table: {table_id}\n")

files = [f for f in os.listdir(SOURCE_DIR) if f.endswith('.csv')]

if not files:
    print(" No CSV files found in the source directory.")

for file in files:
    full_filename = os.path.splitext(file)[0]
    campaign_name = full_filename
    company_name = get_company_name(full_filename)
    file_path = os.path.join(SOURCE_DIR, file)
    delete_query = f"DELETE FROM `{table_id}` WHERE campaign_name = '{campaign_name}'"
    try:
        client.query(delete_query).result()
        print(f"Clearing old records for → {campaign_name} (Company: {company_name})")
    except Exception as e:
        pass

    try:
        df = pd.read_csv(file_path, dtype=str)

        clean_df = pd.DataFrame()

        clean_df['phone_number']    = df['Phone Number']
        clean_df['status']          = df['Status']
        clean_df['attempts']        = pd.to_numeric(df['Attempts Count'], errors='coerce').fillna(0).astype(int)
        clean_df['duration_sec']    = pd.to_numeric(df['Last Call Duration (seconds)'], errors='coerce').fillna(0).astype(int)
        clean_df['dtmf_choice']     = df['DTMF Choice']
        clean_df['hangup_cause']    = df['Hangup Cause']
        clean_df['created_at']      = pd.to_datetime(df['Created At'])
        clean_df['last_call_at']    = pd.to_datetime(df['Last Call At'])

        clean_df['campaign_name']   = campaign_name
        clean_df['company_name']    = company_name
        clean_df['ingestion_time']  = pd.Timestamp.now()

        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
        client.load_table_from_dataframe(clean_df, table_id, job_config=job_config).result()

        print(f"Success: {file} uploaded.\n")

    except Exception as e:
        print(f"Error processing {file}: {e}")

print("---")
print("Master Update Complete! Your Looker Studio dashboard is now updated with the latest company-segmented data.")

📂 Scanning for updates in: /content/drive/MyDrive/Portfolio Attachments/AICallsSystemData_copy
📤 Target table: e-cycling-485018-t0.callva_data.master_campaign_analytics_copy



KeyboardInterrupt: 